In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import joblib

# 1. โหลดข้อมูลดิบชุดใหม่
file_name = 'zeus_dataset_combined_new.csv'
print(f"🚀 1. กำลังโหลดข้อมูลใหม่จาก: {file_name}")
df = pd.read_csv(file_name)

# 2. ทำความสะอาดและสร้าง Feature ใหม่ (Feature Engineering)
print("🧼 2. กำลังทำความสะอาดข้อมูล (Data Cleaning)...")
# แปลงคอลัมน์ datetime ให้คอมพิวเตอร์รู้จักว่าเป็น 'เวลา' จริงๆ
df['datetime'] = pd.to_datetime(df['datetime'])

# ดึง 'ชั่วโมง' ออกมา
df['hour'] = df['datetime'].dt.hour

# ระบุกลางวัน/กลางคืน (6 โมงเช้า - 6 โมงเย็น คือกลางวัน)
df['is_day'] = df['hour'].apply(lambda x: 1 if 6 <= x <= 18 else 0)

# ทำความสะอาดข้อมูล 'ฝน' (ตัด Noise)
# ถ้าฝนตกน้อยกว่า 0.05 มม. ให้ถือว่าไม่ตก (0) เพื่อไม่ให้ AI สับสน
df['rain_clean'] = df['rain'].apply(lambda x: 1 if x > 0.05 else 0)

# 3. เลือกเฉพาะคอลัมน์ที่หน้าเว็บเราใช้
features_to_keep = ['temp', 'humidity', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day', 'rain_clean']
df = df[features_to_keep]

print(f"✅ ข้อมูลพร้อมใช้งาน! จำนวนทั้งหมด {len(df)} แถว")
print("\n🧠 3. กำลังส่งเทพ Zeus เข้าโรงเรียน (Training AI Models)...")

# --- Model 1: The Oracle (อุณหภูมิ) ---
print("   ⏳ กำลังฝึก: โมเดลอุณหภูมิ (Temperature)...")
X_temp = df[['humidity', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day']]
y_temp = df['temp']
model_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_temp.fit(X_temp, y_temp)
joblib.dump(model_temp, 'zeus_oracle_model.pkl')

# --- Model 2: ความชื้น (Humidity) ---
print("   ⏳ กำลังฝึก: โมเดลความชื้น (Humidity)...")
X_hum = df[['temp', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day']]
y_hum = df['humidity']
model_hum = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_hum.fit(X_hum, y_hum)
joblib.dump(model_hum, 'zeus_humidity_model.pkl')

# --- Model 3: โอกาสเกิดฝน (Rain Probability) ---
print("   ⏳ กำลังฝึก: โมเดลทำนายฝน (Rain Classifier)...")
X_rain = df[['temp', 'humidity', 'pressure', 'uv', 'wind_speed', 'hour', 'is_day']]
y_rain = df['rain_clean']
model_rain = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rain.fit(X_rain, y_rain)
joblib.dump(model_rain, 'zeus_rain_class_model.pkl')

# --- Model 4: รังสี UV ---
print("   ⏳ กำลังฝึก: โมเดลรังสี UV...")
X_uv = df[['temp', 'humidity', 'pressure', 'rain', 'wind_speed', 'hour', 'is_day']]
y_uv = df['uv']
model_uv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_uv.fit(X_uv, y_uv)
joblib.dump(model_uv, 'zeus_uv_model.pkl')

print("\n🎉 เสร็จสมบูรณ์! สมองของ Zeus ได้รับการอัปเกรดด้วยข้อมูลใหม่เรียบร้อยแล้ว!")


🚀 1. กำลังโหลดข้อมูลใหม่จาก: zeus_dataset_combined_new.csv
🧼 2. กำลังทำความสะอาดข้อมูล (Data Cleaning)...
✅ ข้อมูลพร้อมใช้งาน! จำนวนทั้งหมด 9475 แถว

🧠 3. กำลังส่งเทพ Zeus เข้าโรงเรียน (Training AI Models)...
   ⏳ กำลังฝึก: โมเดลอุณหภูมิ (Temperature)...
   ⏳ กำลังฝึก: โมเดลความชื้น (Humidity)...
   ⏳ กำลังฝึก: โมเดลทำนายฝน (Rain Classifier)...
   ⏳ กำลังฝึก: โมเดลรังสี UV...

🎉 เสร็จสมบูรณ์! สมองของ Zeus ได้รับการอัปเกรดด้วยข้อมูลใหม่เรียบร้อยแล้ว!
